In [1]:
# ✅ Install required packages (run only once)
!pip install -U insightface onnxruntime opencv-python

# ✅ Imports
import os
import cv2
import numpy as np
import pickle
from numpy.linalg import norm
from insightface.app import FaceAnalysis


In [2]:
# 🔍 Cosine similarity function
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b)))

# 🎯 Match face from DB
def match_face(embedding, face_db, threshold=0.5):
    best_name = "Unknown"
    best_score = 0
    for name, db_emb in face_db:
        sim = cosine_similarity(embedding, db_emb)
        if sim > best_score:
            best_score = sim
            best_name = name
    if best_score < threshold:
        best_name = "Unknown"
    return best_name, best_score


In [ ]:
# 🏗️ Build and save face database
def build_face_database(folder_path="database", save_path="face_database.pkl"):
    app = FaceAnalysis(name='buffalo_s', providers=['CPUExecutionProvider'])
    app.prepare(ctx_id=0)

    database = []
    for person_name in os.listdir(folder_path):
        person_dir = os.path.join(folder_path, person_name)
        if not os.path.isdir(person_dir):
            continue
        for file in os.listdir(person_dir):
            img_path = os.path.join(person_dir, file)
            img = cv2.imread(img_path)
            if img is None:
                continue
            faces = app.get(img)
            if not faces:
                continue
            emb = faces[0].embedding
            database.append((person_name, emb))
            print(f"✅ Added {person_name}")
            break  # Only one image per person

    with open(save_path, "wb") as f:
        pickle.dump(database, f)
        print(f"✅ Saved embeddings for {len(database)} people to {save_path}")

# Run this once after uploading your dataset
build_face_database("database/database")



In [ ]:
!unzip test_dataset.zip -d test_dataset

In [ ]:
import os
import cv2
import pickle
from insightface.app import FaceAnalysis

# Load face database
with open("face_database.pkl", "rb") as f:
    face_db = pickle.load(f)

# Initialize face detector
app = FaceAnalysis(name='buffalo_s', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0)

test_folder = "test_dataset/test dataset"   # ✅ Corrected path
output_folder = "results"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(test_folder):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(test_folder, filename)
    print(f"\n📷 Processing: {img_path}")
    img = cv2.imread(img_path)

    if img is None:
        print(f"❌ Failed to read {filename}")
        continue

    faces = app.get(img)
    if not faces:
        print(f"⚠️ No face detected in {filename}")
        continue

    for face in faces:
        name, score = match_face(face.embedding, face_db)
        box = face.bbox.astype(int)
        cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        cv2.putText(img, f"{name} ({score:.2f})", (box[0], box[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, img)
    print(f"✅ Saved to: {save_path}")


In [ ]:
import os
import cv2
from PIL import Image
from IPython.display import display

# 📁 Folder where output images are saved
results_folder = "results"

# 🔁 Loop through all result images
for filename in os.listdir(results_folder):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    
    image_path = os.path.join(results_folder, filename)
    img = cv2.imread(image_path)

    if img is None:
        print(f"❌ Could not read {filename}")
        continue

    # Convert BGR to RGB for correct color in Jupyter
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convert to PIL Image for display
    pil_img = Image.fromarray(img_rgb)

    # Optional: Resize large images for display
    max_width = 800
    if pil_img.width > max_width:
        ratio = max_width / pil_img.width
        new_size = (max_width, int(pil_img.height * ratio))
        pil_img = pil_img.resize(new_size)

    print(f"🖼️ Displaying: {filename}")
    display(pil_img)
